## Setup and imports

In [ ]:
EXP_NAME = "28APR_add_presprop_SPS"
dataset = "28APR_add_presprop"
feature_map = "features_synth_with_ind"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{dataset}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

In [ ]:
# sps_audit.head()

# Basics

In [ ]:
scorings = sps_audit.groupby('feature')[['sensitivity_scoring', 'fidelity_scoring']].first()

sps_audit = sps_audit.drop(['sensitivity_scoring', 'fidelity_scoring'], axis=1)

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['roc_auc']].count()
iteration_per_feature

# Feature-wise analysis

### Counterfactual sensitivity

In [ ]:
sps_audit.groupby('feature')['cf_sensitivity'].agg(['mean','std','median']).sort_values(by='median')

In [ ]:
# Counterfactual sensitivity
f, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=sps_audit, x="feature", y="cf_sensitivity", showmeans=True, 
            meanprops={"marker": "+",
                       "markeredgecolor": "black",
                       "markersize": "8"}, ax=ax)
plt.title('Counterfactual Sensitivity by feature')
plt.ylabel('')
plt.xlabel('Feature')
plt.plot()


## Trade-off analysis

In [ ]:
group_ieco_mace_cols = sps_audit.columns.str.startswith('ieco_mace_')
group_ieco_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('ieco_mace_')
sps_audit['max_ieco_mace'] = sps_audit.loc[:, group_ieco_mace_cols].max(axis=1)
sbs_audit_baseline['max_ieco_mace'] = sbs_audit_baseline.loc[:, group_ieco_mace_baseline_cols].max(axis=1)

group_total_mace_cols = sps_audit.columns.str.startswith('total_mace_')
group_total_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('total_mace_')
sps_audit['max_total_mace'] = sps_audit.loc[:, group_total_mace_cols].max(axis=1)
sbs_audit_baseline['max_total_mace'] = sbs_audit_baseline.loc[:, group_total_mace_baseline_cols].max(axis=1)

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)

feature_desc_stats = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['feature'])[['te_error', 'max_total_mace', 'auprc']].median()
feature_corr_stats = full_audit[full_audit['bucket'] == 'x_corr'].groupby(['feature'])[['te_error', 'max_total_mace', 'auprc']].median()

In [ ]:
# TE Error Delta: 
# > 0: placing the feature in Xdesc degrades causal fidelity
feature_desc_stats['te_error_delta'] = (feature_desc_stats['te_error'] - feature_corr_stats['te_error']) / feature_corr_stats['te_error']

# Max MACE Delta: 
# > 0: placing the feature in Xdesc worsens bias for the worse off group
feature_desc_stats['max_mace_delta'] = (feature_desc_stats['max_total_mace'] - feature_corr_stats['max_total_mace']) / feature_corr_stats['max_total_mace']

# AUPRC Delta: 
# < 0: placing the feature in Xdesc degrades the utility of the model
feature_desc_stats['auprc_delta'] = (feature_desc_stats['auprc'] - feature_corr_stats['auprc']) / feature_corr_stats['auprc']

## Fairness elasticity: 
# FEU > 1 => highly elastic, i.e. big relative reduction in bias for a small relative cost in utility
# 0 < FEU < 1 => inelastic, i.e. the relative cost in utility is greater than the relative reduction in bias
# FEU <0 => Free lunch, i.e. bias is reduced and the performance is improved, OR lose-lose, i.e. bias is increased and the performance is degraded
feature_desc_stats['fairness_elasticity'] = feature_desc_stats['max_mace_delta'] / feature_desc_stats['auprc_delta'] 
feature_desc_stats['Xdesc candidate'] = ((feature_desc_stats['fairness_elasticity'] > 1) | (feature_desc_stats['fairness_elasticity'] < 0)) & (feature_desc_stats['max_mace_delta'] < 0)

analysis_cols = ['te_error_delta','max_mace_delta', 'auprc_delta', 'fairness_elasticity', 'Xdesc candidate']

print(feature_desc_stats[analysis_cols].sort_values('fairness_elasticity', ascending=False).to_markdown())

In [ ]:
feature_desc_stats.sort_values('auprc_delta', ascending=False, inplace=True)
pareto_frontier = []
current_min_te_error_redux = -1

print('--- Features on the TE error reduction / AUPRC Delta Frontier ---')
for idx, solution in feature_desc_stats.iterrows():
  if solution['te_error_delta'] > current_min_te_error_redux:
    pareto_frontier.append(solution)
    current_min_te_error_redux = solution['te_error_delta']
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df[analysis_cols].to_markdown())

In [ ]:
feature_desc_stats.sort_values('auprc_delta', ascending=False, inplace=True)
pareto_frontier = []
current_max_mace_redux = -1

print('--- Features on the MACE reduction / AUPRC Delta Frontier ---')
for idx, solution in feature_desc_stats.iterrows():
  if solution['max_mace_delta'] > current_max_mace_redux:
    pareto_frontier.append(solution)
    current_max_mace_redux = solution['max_mace_delta']
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df[analysis_cols].to_markdown())

In [ ]:
feature_desc_stats.sort_values('max_mace_delta', ascending=False, inplace=True)
pareto_frontier = []
current_min_te_error_redux = -1

print('--- Features on the TE error reduction / MACE reduction Frontier ---')
for idx, solution in feature_desc_stats.iterrows():
  if solution['te_error_delta'] > current_min_te_error_redux:
    pareto_frontier.append(solution)
    current_min_te_error_redux = solution['te_error_delta']
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df[analysis_cols].to_markdown())

# Config-level trade-off analyses

In [ ]:
baseline_te_error = sbs_audit_baseline['te_error'].values[0]
baseline_total_mace = sbs_audit_baseline['total_mace'].values[0]
baseline_max_total_mace = sbs_audit_baseline['max_total_mace'].values[0]
baseline_roc_auc = sbs_audit_baseline['roc_auc'].values[0]
baseline_auprc = sbs_audit_baseline['auprc'].values[0]

## Total Effect estimation vs Fairness trade-off

In [ ]:
print(f"Baseline TE error: {baseline_te_error}")
print(f"Baseline MACE: {baseline_total_mace}")
print(f"Baseline maximum group MACE: {baseline_max_total_mace}")

In [ ]:
te_error_vs_fairness = full_audit.groupby('iteration')[['te_error', 'max_total_mace', 'auprc']].first().sort_values('max_total_mace')
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

pareto_frontier = []
current_min_te_error = 1

print('--- Configurations on the TE error / MACE Pareto Frontier ---')
for idx, solution in te_error_vs_fairness.iterrows():
  if solution['te_error'] < current_min_te_error:
    pareto_frontier.append(solution)
    current_min_te_error = solution['te_error']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs[idx]}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


In [ ]:


fig, axes = plt.subplots(1, 2,  figsize=(20, 6))
axes[0].plot(baseline_te_error, baseline_max_total_mace, marker="D", color="#D92B89", linestyle="")
axes[1].plot(baseline_te_error, baseline_max_total_mace, marker="D", color="#D92B89", linestyle="")
sns.scatterplot(data=te_error_vs_fairness, x='te_error', y='max_total_mace', size="auprc", sizes=(20, 200) ,ax=axes[0])
axes[0].legend_.set_title('AUPRC')
sns.lineplot(data=pareto_frontier_df, x='te_error', y='max_total_mace', color="green", marker='', linestyle="--", errorbar=None, ax=axes[0])
sns.lineplot(data=pareto_frontier_df, x='te_error', y='max_total_mace', color="green", linestyle="--", errorbar=None, ax=axes[1])
sns.scatterplot(data=feature_desc_stats, x='te_error', y='max_total_mace', hue='feature', color='', ax=axes[1])
plt.xlabel('TE Error')
plt.ylabel('Max group MACE')
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   "Feature config results",
                   ]+feature_desc_stats.index.to_list(),
           loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=te_error_vs_fairness, x='te_error', y='max_total_mace', size="auprc", sizes=(10, 300) ,ax=ax)
ax.plot(baseline_te_error, baseline_max_total_mace, marker="D", color="#D92B89", linestyle="", label='Bias-unaware configuration')
sns.lineplot(data=pareto_frontier_df, x='te_error', y='max_total_mace', color="green", marker='', linestyle="--", errorbar=None, label='Pareto frontier', ax=ax)
plt.xlabel('TE Error')
plt.ylabel('Max group MACE')
plt.legend(title="AUPRC", alignment='left', loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")
ax.set_title("Feature pathway configurations analysis", fontsize="large", fontweight="bold", pad=15)
plt.show()

## Utility vs Fairness trade-off

In [ ]:
print(f"Baseline Average Precision: {baseline_auprc}")
print(f"Baseline ROC AUC: {baseline_roc_auc}")
print(f"Baseline max MACE: {baseline_max_total_mace}")

In [ ]:
utility_vs_fairness = full_audit.groupby('iteration')[['te_error', 'max_total_mace', 'auprc']].first().sort_values('max_total_mace')
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_baseline = len(positives) / len(dataset)
print(f'Chance-level AUPRC: {auprc_baseline}')

pareto_frontier = []
current_max_utility = -1

print('\n--- Configurations on the AUPRC / max group MACE Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['auprc'] > current_max_utility:
    pareto_frontier.append(solution)
    current_max_utility = solution['auprc']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


fig, axes = plt.subplots(1, 2,  figsize=(20, 6))
axes[0].plot(baseline_auprc, baseline_max_total_mace, marker="D", color="#D92B89", linestyle="")
axes[1].plot(baseline_auprc, baseline_max_total_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='auprc', y='max_total_mace', color="green", marker='', linestyle="--", errorbar=None, ax=axes[0])
sns.lineplot(data=pareto_frontier_df, x='auprc', y='max_total_mace', color="green", linestyle="--", errorbar=None, ax=axes[1])
sns.scatterplot(data=utility_vs_fairness, x='auprc', y='max_total_mace', ax=axes[0])
sns.scatterplot(data=feature_desc_stats, x='auprc', y='max_total_mace', hue='feature', color='', ax=axes[1])
# plt.axvline(auprc_baseline)
plt.xlabel('Average Precision (AUPRC)')
plt.ylabel('Max group MACE')
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit'
                   ]+feature_desc_stats.index.to_list(),
           loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")

plt.show()

## Utility vs TE error

In [ ]:
te_error_vs_utility = full_audit.groupby('iteration')[['te_error', 'max_total_mace', 'auprc']].first().sort_values('auprc', ascending=False)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

pareto_frontier = []
current_min_te_error = 1

print('--- Configurations on the TE error / Utility Pareto Frontier ---')
for idx, solution in te_error_vs_utility.iterrows():
  if solution['te_error'] < current_min_te_error:
    pareto_frontier.append(solution)
    current_min_te_error = solution['te_error']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())

# All Configs explored

In [ ]:
def find_config(row):
  return x_desc_configs.get(row['id'], "empty")

all_configs = utility_vs_fairness.sort_values('max_total_mace').reset_index(names="id")
all_configs['config'] = all_configs.apply(find_config, axis=1)

lowest_te_error = all_configs.sort_values('te_error').reset_index().loc[0, "config"]
lowest_max_total_mace = all_configs.sort_values('max_total_mace').reset_index().loc[0, "config"]
highest_auprc  = all_configs.sort_values('auprc', ascending=False).reset_index().loc[0, "config"]
print(f'Config with the lowest TE error: {lowest_te_error}\n')
print(f'Config with the lowest max MACE: {lowest_max_total_mace}\n')
print(f'Config with the highest AUPRC: {highest_auprc}\n')
print(all_configs.to_markdown())

In [ ]:
print('\n--- All Explored Configurations ---')
for idx, solution in utility_vs_fairness.iterrows():
  flag = ""
  overlap = 0
  # if "age" in x_desc_configs[idx]:
  #   overlap += 1
  # if "symptom_1_obs" in x_desc_configs[idx]:
  #   overlap += 1
  # if 'symptom_3_obs' in x_desc_configs[idx]:
  #   overlap += 1
  # if 'biomarker_1_obs' in x_desc_configs[idx]:
  #   overlap += 1
  # if 'procedure' in x_desc_configs[idx]:
  #   overlap += 1
  # if overlap > 1 and len(x_desc_configs[idx]) <= overlap + 1:
  #   flag = f"{overlap}-FEATURE OVERLAP => "
  # if overlap == 5 and len(x_desc_configs[idx]) == 5:
  #   flag = "ORIGINAL DESC FEATURES => "

  print(f'{flag}Iteration {idx}, Xdesc: {x_desc_configs.get(idx, "empty")}')